# 📊 SÍNTESE VISUAL 3 — pAUC@0.1 × Score DCASE × Multi-SNR

In [ ]:
import json, sys, numpy as np
sys.path.insert(0, '..')
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

METRICS_DIR = Path('experiments_results/metrics')
FIGURES_DIR = Path('experiments_results/figures')

plt.rcParams.update({
    'figure.facecolor':'#0d1117','axes.facecolor':'#161b22',
    'text.color':'#f0f6fc','axes.labelcolor':'#f0f6fc',
    'xtick.color':'#8b949e','ytick.color':'#8b949e',
    'axes.edgecolor':'#30363d','grid.color':'#30363d','grid.alpha':0.5,
})

def load(nb_id):
    p = METRICS_DIR/f'{nb_id}_results.json'
    return json.load(open(p)) if p.exists() else {}

def best_pauc(data):
    for key in ['avg_prot_a','pauc01']:
        if key in data: return float(data[key])
    if 'protocol_a' in data: return float(data['protocol_a'].get('pauc01',0))
    if 'spec' in data: return float(data['spec'].get('avg_prot_a',0))
    if 'models' in data:
        vals = [v.get('pauc01',0) for v in data['models'].values() if isinstance(v,dict)]
        if vals: return float(max(vals))
    if 'model' in data:
        m = data['model']
        if isinstance(m,dict): return float(m.get('pauc01',0))
    if 'ranking' in data:
        vals = [v.get('pauc01',0) for v in data['ranking'].values()]
        if vals: return float(max(vals))
    return 0.0

all_data = {f'nb{i:02d}': load(f'nb{i:02d}') for i in range(1,29)}
print(f"✅ {sum(1 for v in all_data.values() if v)} resultados carregados")


In [ ]:
# Score DCASE por domínio (NB28)
nb28 = load('nb28')
spec = nb28.get('spec', {})
per_domain = spec.get('per_domain', {})

if per_domain:
    domains = list(per_domain.keys())
    pa = [per_domain[d].get('prot_a_pauc',0) for d in domains]
    pb = [per_domain[d].get('prot_b_pauc',0) for d in domains]
    dc = [per_domain[d].get('dcase_score',0) for d in domains]

    x = np.arange(len(domains))
    fig, axes = plt.subplots(1,2,figsize=(14,5))
    for ax in axes: ax.set_facecolor('#161b22')

    axes[0].bar(x-0.25, pa, 0.25, label='Protocolo A', color='#2ea043', alpha=0.85)
    axes[0].bar(x,      dc, 0.25, label='Score DCASE', color='#f0883e', alpha=0.85)
    axes[0].bar(x+0.25, pb, 0.25, label='Protocolo B', color='#1f6feb', alpha=0.85)
    axes[0].axhline(0.80,color='white',ls='--',lw=1.2,alpha=0.6)
    axes[0].set_xticks(x); axes[0].set_xticklabels([d.upper() for d in domains])
    axes[0].set_ylim(0,1.05); axes[0].set_ylabel('pAUC@0.1')
    axes[0].set_title('NB28 — Score Final por Domínio (LOSO)', fontsize=11, fontweight='bold')
    axes[0].legend(fontsize=9); axes[0].grid(True,axis='y',alpha=0.3)

    # Multi-SNR
    nb25 = load('nb25')
    snr_data = nb25.get('per_snr', {})
    if snr_data:
        snrs = sorted([v.get('snr_db',0) for v in snr_data.values()])
        pa25 = [v.get('prot_a',0) for v in sorted(snr_data.values(), key=lambda x:x.get('snr_db',0))]
        pb25 = [v.get('prot_b',0) for v in sorted(snr_data.values(), key=lambda x:x.get('snr_db',0))]
        axes[1].plot(snrs, pa25, 'o-', color='#2ea043', lw=2, ms=8, label='Protocolo A')
        axes[1].plot(snrs, pb25, 's--', color='#1f6feb', lw=2, ms=8, label='Protocolo B')
        axes[1].axhline(0.80,color='#f0883e',ls='--',lw=1.5,label='Limite 0.80')
        axes[1].set_xlabel('SNR (dB)'); axes[1].set_ylabel('pAUC@0.1')
        axes[1].set_title('NB25 — MIMII Multi-SNR', fontsize=11, fontweight='bold')
        axes[1].legend(fontsize=9); axes[1].grid(True,alpha=0.3); axes[1].set_ylim(0,1.05)

    fig.patch.set_facecolor('#0d1117')
    plt.tight_layout()
    plt.savefig('experiments_results/figures/visual3_dcase_snr.png', dpi=130,
                bbox_inches='tight', facecolor='#0d1117')
    plt.show()
else:
    print("Dados NB28 não disponíveis — rode NB28 primeiro.")
